<a href="https://colab.research.google.com/github/dannyngweekiat/Introduction-to-Machine-Vision/blob/main/notebooks/08_mini_ai_vision_project.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a> <a href="https://github.com/dannyngweekiat/Introduction-to-Machine-Vision/blob/main/notebooks/08_mini_ai_vision_project.ipynb" target="_blank"><img src="https://img.shields.io/badge/View%20on-GitHub-181717?style=flat-square&amp;logo=github&amp;logoColor=white" alt="View on GitHub"></a>

# Lesson 8: Mini AI Vision Project

## Your mission
Choose one simple vision project, customise it, test it fairly, and explain a limitation.

**Key words:** test · customise · limitation

Each project below is independent. Pick one section, run its code cell, then make one change at a time.


## Choose a project section

- **Project A — Colour mask:** find pixels of a chosen colour.
- **Project B — Digit recogniser:** predict handwritten digits and inspect mistakes.
- **Project C — Image classifier:** compare a model’s top three labels.
- **Project D — Object detector:** find objects and draw predicted boxes.

You do not need to change a project switch. Run only the section you want to explore.


## Project A: Colour mask

Find the orange-brown pixels in a coffee photograph.


In [ ]:
# Import OpenCV for colour conversion and masking.
import cv2
# Import NumPy for number arrays.
import numpy as np
# Import Matplotlib to display the results.
import matplotlib.pyplot as plt
# Load a ready-made sample photograph.
from skimage import data

# Choose a colour image with orange-brown coffee beans.
image = data.coffee()
# Convert RGB colours into HSV, which is easier to select by colour.
hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
# Set the lowest orange-brown HSV values to keep.
lower = np.array([5, 80, 50])
# Set the highest orange-brown HSV values to keep.
upper = np.array([30, 255, 255])
# Make a black-and-white mask for pixels inside the chosen colour range.
mask = cv2.inRange(hsv, lower, upper)
# Keep only the pixels selected by the mask.
result = cv2.bitwise_and(image, image, mask=mask)

# Create three side-by-side spaces for a clear comparison.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# Show the original photograph.
axes[0].imshow(image)
# Label the original photograph.
axes[0].set_title('Original image')
# Show the black-and-white selection mask.
axes[1].imshow(mask, cmap='gray')
# Label the selection mask.
axes[1].set_title('Selected colour mask')
# Show the selected pixels on their own.
axes[2].imshow(result)
# Label the final colour result.
axes[2].set_title('Pixels kept by the mask')
# Repeat the same tidy-up step for every plot.
for ax in axes:
    # Hide axes because they do not help us read this image.
    ax.axis('off')
# Space the plots so titles stay readable.
fig.tight_layout()
# Show how many pixels matched the colour rule.
print('Highlighted pixels:', np.count_nonzero(mask))


## Project B: Handwritten-digit recogniser

Train a small model, then compare its predictions with the true digit labels.


In [ ]:
# Import Matplotlib to show the digit predictions.
import matplotlib.pyplot as plt
# Load the built-in handwritten-digit data.
from sklearn.datasets import load_digits
# Split examples into training and fair-test groups.
from sklearn.model_selection import train_test_split
# Import the nearby-neighbour voting model.
from sklearn.neighbors import KNeighborsClassifier
# Import a tool for measuring prediction accuracy.
from sklearn.metrics import accuracy_score

# Load labelled 8 by 8 digit pictures.
digits = load_digits()
# Keep one quarter of the examples secret for testing.
X_train, X_test, y_train, y_test = train_test_split(digits.data, digits.target, test_size=0.25, random_state=12, stratify=digits.target)
# Let five similar training examples vote for each prediction.
model = KNeighborsClassifier(n_neighbors=5)
# Train the model using the labelled examples.
model.fit(X_train, y_train)
# Predict the secret test examples.
predicted = model.predict(X_test)
# Calculate the percentage of correct predictions.
accuracy = accuracy_score(y_test, predicted)
# Print the score before looking at individual examples.
print('Test accuracy:', f'{accuracy:.1%}')

# Create six small plots to make the predictions easy to inspect.
fig, axes = plt.subplots(1, 6, figsize=(9, 2.2))
# Repeat the display steps for six test digits.
for ax, picture, truth, guess in zip(axes, X_test[:6].reshape(-1, 8, 8), y_test[:6], predicted[:6]):
    # Draw the digit with crisp pixel edges.
    ax.imshow(picture, cmap='gray_r', interpolation='nearest')
    # Use green for a correct prediction and red for a mistake.
    colour = 'green' if truth == guess else 'crimson'
    # Label each picture with the real and predicted digit.
    ax.set_title(f'True {truth}\nAI {guess}', color=colour, fontsize=9)
    # Hide axes because they do not help us read a digit.
    ax.axis('off')
# Space the digit plots so labels stay readable.
fig.tight_layout()


## Project C: Image classifier

This project downloads a pre-trained model on its first run, so Google Colab needs an internet connection.


In [ ]:
# Import OpenCV for image resizing.
import cv2
# Import NumPy for image arrays.
import numpy as np
# Import Matplotlib to show the image and scores.
import matplotlib.pyplot as plt
# Load a ready-made sample image.
from skimage import data
# Import the pre-trained classifier and its helper tools.
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions

# Choose a sample picture for the classifier.
image = data.chelsea()
# Load the model weights learned from many labelled pictures.
model = MobileNetV2(weights='imagenet')
# Resize the picture to the size expected by this model.
model_input = cv2.resize(image, (224, 224)).astype('float32')
# Prepare the image numbers and add one image batch dimension.
model_input = preprocess_input(model_input[None, ...])
# Ask for the three strongest label predictions.
predictions = decode_predictions(model.predict(model_input, verbose=0), top=3)[0]
# Turn machine-style labels into labels that are easier to read.
labels = [label.replace('_', ' ') for _, label, _ in predictions]
# Keep the three confidence scores for the chart.
scores = [score for _, _, score in predictions]

# Create one space for the picture and one for the score chart.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Show the image the model examined.
axes[0].imshow(image)
# Label the source image.
axes[0].set_title('Image sent to the model')
# Hide axes because they do not help us read the image.
axes[0].axis('off')
# Draw a chart of the three most likely labels.
axes[1].barh(labels[::-1], scores[::-1], color='#3D8DFF')
# Keep confidence values on a clear 0 to 1 scale.
axes[1].set_xlim(0, 1)
# Label the chart axis.
axes[1].set_xlabel('Model confidence')
# Label the score chart.
axes[1].set_title('Top three suggestions')
# Space the picture and chart neatly.
fig.tight_layout()


## Project D: Object detector

This project downloads a small detector on its first run, so Google Colab needs an internet connection.


In [ ]:
# Install the object-detection library in this Colab session.
!pip -q install ultralytics
# Import Matplotlib to show the detection result.
import matplotlib.pyplot as plt
# Load a ready-made sample image.
from skimage import data
# Import the YOLO object detector.
from ultralytics import YOLO

# Choose an image with a person for the detector to find.
image = data.astronaut()
# Load a small pre-trained object detector.
model = YOLO('yolov8n.pt')
# Keep predictions with at least 35 percent confidence.
results = model(image, conf=0.35, verbose=False)
# Draw the model's boxes and labels on a copy of the image.
annotated_bgr = results[0].plot()
# Convert OpenCV's blue-green-red order back to red-green-blue.
annotated_rgb = annotated_bgr[:, :, ::-1]

# Create a large display area for the predicted boxes.
plt.figure(figsize=(7, 7))
# Show the annotated detection image.
plt.imshow(annotated_rgb)
# Label the detection result.
plt.title('Object-detector predictions')
# Hide axes because they do not help us inspect the boxes.
plt.axis('off')
# Show the image with its predicted boxes.
plt.show()

# Get the readable names for each predicted class.
names = results[0].names
# Read the class number for each predicted box.
class_ids = results[0].boxes.cls.cpu().numpy().astype(int)
# Read the confidence score for each predicted box.
confidences = results[0].boxes.conf.cpu().numpy()
# Print a clear list of the objects the detector found.
for class_id, confidence in zip(class_ids, confidences):
    # Display one object label and its confidence score.
    print(f'{names[class_id]}: {confidence:.1%}')


## Your project record

In a Markdown cell below, complete:

1. **Project title:**
2. **What I changed:**
3. **Three tests and results:**
4. **My best output:**
5. **One limitation:**

Prepare a 1–2 minute explanation. Show what your code does, not just whether it worked.


In [ ]:
# YOUR TURN
# Pick one project section above and run its code cell before changing anything.
# Hint 1: Change one setting at a time, such as a colour range, number of neighbours, image, or confidence threshold.
# Hint 2: Run three fair tests and write down what changed in each test.
# Hint 3: Use the visible image, chart, labels, or boxes as evidence for your explanation.
# Check: Save or screenshot your best result and write one thing that still needs improving.


## Final reflection

What would you need to improve before using your project in the real world?
